In [8]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver,MemorySaver  # both are same


In [9]:
llm = ChatOllama(model='qwen2.5:0.5b-instruct',temperature=0.2,max_token=100)

In [10]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]
    


In [11]:
def chat_node(state:ChatState):
    messages = state['messages']
    res = llm.invoke(messages)
    return {'messages':[res.content]}


In [12]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)


In [13]:
thread_id = '1'
while True:
    user_message = input("Ask anything: ")
    print("Human : ",user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    config = {'configurable':{'thread_id':thread_id}}
    res = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print('AI : ',res['messages'][-1].content)

Human :  hi
AI :  Hello! How can I assist you today?
Human :  my name is girish
AI :  Hello Girish! As an AI language model, I'm here to help answer your questions and provide information on various topics. How may I assist you today?
Human :  what is my name
AI :  Your name is "Girish".
Human :  exit


In [7]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='', additional_kwargs={}, response_metadata={}, id='1902d566-f21e-4a66-ab38-341eebd89ed1'), HumanMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={}, id='210e4565-c959-4f55-9f58-96291ab75ca6'), HumanMessage(content='hio', additional_kwargs={}, response_metadata={}, id='f40537d0-8759-4c48-8053-9f36304418f5'), HumanMessage(content="Hello! I'm here to help with any questions or topics you'd like to discuss. What would you like to know?", additional_kwargs={}, response_metadata={}, id='f8a03345-1dab-46d6-b81b-9ce4d426c72c'), HumanMessage(content='my name is girish', additional_kwargs={}, response_metadata={}, id='01d89cc1-8ec5-4f77-b5e6-25e79b348c83'), HumanMessage(content="Hello Girish! As an AI language model, I don't have a physical presence, so I can't physically interact with you in the same way that a human might. However, I'm here to assist you with any questions or topics you'd like